# Matplotlib Phase 2: Distribution & Outlier Detection
### Credit Card Risk Analysis Project

Real credit data is messy: a reported income of $0, an applicant age of 150, a
debt-to-income ratio of 200%. Before you clean any of that, you need to **see** it —
which is exactly what histograms and boxplots are for.

This notebook covers 2 topics:
5. **Histograms** — visualizing the frequency distribution of continuous data
6. **Boxplots** — spotting outliers via spread, quartiles, and whiskers

**Format:** Each question has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Try your own answer before peeking!

Run the setup cell below first — it builds a synthetic applicant pool that deliberately
includes messy, unrealistic values, the same way real credit bureau data would.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

n = 600

# Credit scores: mostly a realistic 300-850 range, with a few bad/erroneous entries mixed in
credit_scores = np.random.normal(loc=680, scale=60, size=n)
credit_scores = np.clip(credit_scores, 300, 850)
credit_scores = np.concatenate([credit_scores, [300, 305, 845, 850]])  # a few extreme-but-valid edge cases

# Applicant age: realistic adults, but with a couple of clearly bad data-entry errors
age = np.random.normal(loc=40, scale=12, size=n)
age = np.clip(age, 18, 85)
age = np.concatenate([age, [150, -5]])  # data entry errors: impossible ages

# Annual income: right-skewed (typical of income data), with a couple of $0 entries
annual_income = np.random.lognormal(mean=10.8, sigma=0.5, size=n)
annual_income = np.concatenate([annual_income, [0, 0]])  # missing/misreported income

# Debt-to-income ratio (%): mostly reasonable, but some applicants report impossible DTI
debt_to_income = np.random.normal(loc=35, scale=10, size=n)
debt_to_income = np.clip(debt_to_income, 0, 80)
debt_to_income = np.concatenate([debt_to_income, [150, 180, 200]])  # impossible DTI > 100%

# Risk tier and employment type, for grouped comparisons later
n_rows = len(debt_to_income)
risk_tier = np.random.choice(['Low', 'Medium', 'High'], size=n_rows, p=[0.4, 0.4, 0.2])
employment_type = np.random.choice(['Salaried', 'Self-Employed', 'Unemployed'],
                                    size=n_rows, p=[0.6, 0.3, 0.1])

# Every synthetic array was built slightly differently (some have a couple of
# extra deliberately-bad rows appended), so trim/resize them all to the same
# length before assembling the DataFrame.
credit_scores = np.resize(credit_scores, n_rows)
age = np.resize(age, n_rows)
annual_income = np.resize(annual_income, n_rows)

# Whether the applicant was approved (for a later comparison histogram)
approved = np.where(credit_scores > 620, 1, 0)

applicants = pd.DataFrame({
    'Credit_Score': credit_scores,
    'Age': age,
    'Annual_Income': annual_income,
    'Debt_to_Income': debt_to_income,
    'Risk_Tier': risk_tier,
    'Employment_Type': employment_type,
    'Approved': approved
})

print(applicants.shape)
applicants.head()


---
## Section 5: Histograms

A histogram bins continuous data and shows you how many observations fall into each
bin — the fastest way to answer "what does this variable's distribution actually look
like?" before you commit to any cleaning or modeling decisions.


**Q1.** Create a Figure/Axes pair and plot a histogram of `applicants['Credit_Score']` using `ax.hist()`.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'])
plt.show()


**Q2.** Repeat Q1, but use `bins=30` for a finer-grained view of the distribution's shape.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'], bins=30)
plt.show()


**Q3.** Repeat Q2, adding `edgecolor='black'` so each bin's boundary is visually distinct.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'], bins=30, edgecolor='black')
plt.show()


**Q4.** Repeat Q3, setting the fill color to `'steelblue'` via the `color` parameter.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'], bins=30, edgecolor='black', color='steelblue')
plt.show()


**Q5.** Repeat Q4, and add a title `"Distribution of Applicant Credit Scores"`, xlabel `"Credit Score"`, and ylabel `"Number of Applicants"`.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'], bins=30, edgecolor='black', color='steelblue')
ax.set_title("Distribution of Applicant Credit Scores")
ax.set_xlabel("Credit Score")
ax.set_ylabel("Number of Applicants")
plt.show()


**Q6.** Plot the same histogram with `density=True` instead of the raw count, so the y-axis shows probability density rather than frequency. This is useful when you want to compare distributions of different sample sizes on the same scale.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'], bins=30, edgecolor='black', density=True)
ax.set_title("Credit Score Density")
plt.show()


**Q7.** Split `applicants` into two groups by `Approved` (1 vs 0). Plot both groups' `Credit_Score` histograms on the same Axes with `alpha=0.5` so they're semi-transparent and overlap visibly. Add a legend distinguishing `'Approved'` from `'Denied'`.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
approved_scores = applicants.loc[applicants['Approved'] == 1, 'Credit_Score']
denied_scores = applicants.loc[applicants['Approved'] == 0, 'Credit_Score']

fig, ax = plt.subplots()
ax.hist(approved_scores, bins=25, alpha=0.5, label='Approved')
ax.hist(denied_scores, bins=25, alpha=0.5, label='Denied')
ax.legend()
plt.show()


**Q8.** Plot the `Credit_Score` histogram again, then draw a vertical line at the mean credit score using `ax.axvline()`, with `color='red'` and `linestyle='--'`.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
mean_score = applicants['Credit_Score'].mean()

fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'], bins=30, edgecolor='black')
ax.axvline(mean_score, color='red', linestyle='--')
plt.show()

print(f"Mean credit score: {mean_score:.1f}")


**Q9.** Instead of letting Matplotlib choose bin boundaries automatically, build explicit bin edges with `np.arange(300, 875, 25)` (standard 25-point credit score bands) and pass them to `bins`.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
bin_edges = np.arange(300, 875, 25)

fig, ax = plt.subplots()
ax.hist(applicants['Credit_Score'], bins=bin_edges, edgecolor='black')
ax.set_xlabel("Credit Score (25-point bands)")
plt.show()


**Q10.** Plot a histogram of `applicants['Annual_Income']` with `bins=30`. Note the shape — most income data is right-skewed, with a long tail of high earners.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Annual_Income'], bins=30, edgecolor='black')
ax.set_title("Annual Income Distribution (Right-Skewed)")
plt.show()


**Q11.** Create a Figure with 1 row and 2 columns. On the left Axes, plot the raw `Annual_Income` histogram. On the right Axes, plot a histogram of `np.log1p(applicants['Annual_Income'])` — the log transform is a standard technique for taming skewed financial data before modeling. Give each subplot its own title.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(applicants['Annual_Income'], bins=30, edgecolor='black')
axes[0].set_title("Raw Income")

axes[1].hist(np.log1p(applicants['Annual_Income']), bins=30, edgecolor='black', color='seagreen')
axes[1].set_title("Log-Transformed Income")

plt.show()


**Q12.** Plot a histogram of `applicants['Age']` with `bins=30`. Add `ax.set_xlabel('Age')`. Visually inspect the plot — do you notice any bars sitting far outside a plausible human age range?

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
fig, ax = plt.subplots()
ax.hist(applicants['Age'], bins=30, edgecolor='black')
ax.set_xlabel("Age")
ax.set_title("Applicant Age Distribution")
plt.show()

# The bulk of the distribution sits in a normal adult range, but a value near 150
# (and a negative value near -5) will show up as isolated bars far from the rest —
# a clear visual signal of bad data entry that a .describe() table might not
# immediately draw your eye to.


**Q13 (Capstone).** Create a fully polished histogram of `applicants['Debt_to_Income']`: 30 bins, `edgecolor='black'`, a color of your choice, a bold title, x/y axis labels, and `ax.grid(True, alpha=0.3)`. This is the kind of chart you'd screenshot straight into a data-quality report.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(applicants['Debt_to_Income'], bins=30, edgecolor='black', color='indianred')
ax.set_title("Distribution of Debt-to-Income Ratio", fontsize=13, fontweight='bold')
ax.set_xlabel("Debt-to-Income Ratio (%)")
ax.set_ylabel("Number of Applicants")
ax.grid(True, alpha=0.3)
plt.show()


> **Checkpoint — Section 5:** `ax.hist()` is your go-to for understanding the shape of a
> single continuous variable — its center, spread, skew, and any isolated bars that hint at
> bad data. `bins`, `density`, overlaying groups with `alpha`, and marking the mean with
> `axvline()` are the standard toolkit for making a histogram both accurate and readable.


---
## Section 6: Boxplots (The Outlier Hunter)

A boxplot summarizes a distribution's quartiles and flags anything beyond the whiskers
as an individual point — an outlier. Where a histogram shows you *shape*, a boxplot
shows you *outliers* directly and explicitly, which is exactly what you need when
hunting for impossible debt-to-income ratios or corrupted credit scores.


**Q14.** Create a Figure/Axes pair and plot a basic boxplot of `applicants['Credit_Score']` using `ax.boxplot()`.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(applicants['Credit_Score'])
plt.show()


**Q15.** Plot a boxplot of `applicants['Debt_to_Income']`. Add the title `"Debt-to-Income Ratio: Outlier Check"`. You should see individual dots above the top whisker — the impossible DTI values we deliberately mixed into the data.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(applicants['Debt_to_Income'])
ax.set_title("Debt-to-Income Ratio: Outlier Check")
plt.show()


**Q16.** Repeat Q15, but draw it horizontally using `vert=False` — often easier to read when you have a long axis label.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(applicants['Debt_to_Income'], vert=False)
ax.set_xlabel("Debt-to-Income Ratio (%)")
plt.show()


**Q17.** Repeat Q15, but customize the outlier markers using `flierprops=dict(marker='D', markerfacecolor='red', markersize=6)` so outliers are shown as red diamonds instead of the default circles.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(
    applicants['Debt_to_Income'],
    flierprops=dict(marker='D', markerfacecolor='red', markersize=6)
)
ax.set_title("Debt-to-Income Ratio: Outliers Highlighted")
plt.show()


**Q18.** Build three separate arrays of `Debt_to_Income` values, one per `Risk_Tier` (`'Low'`, `'Medium'`, `'High'`), using boolean filtering. Plot all three as side-by-side boxplots in a single `ax.boxplot()` call by passing a list of the three arrays.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
low_dti = applicants.loc[applicants['Risk_Tier'] == 'Low', 'Debt_to_Income']
medium_dti = applicants.loc[applicants['Risk_Tier'] == 'Medium', 'Debt_to_Income']
high_dti = applicants.loc[applicants['Risk_Tier'] == 'High', 'Debt_to_Income']

fig, ax = plt.subplots()
ax.boxplot([low_dti, medium_dti, high_dti])
plt.show()


**Q19.** Instead of manually filtering three times, build the same grouped list using a loop over `applicants['Employment_Type'].unique()`, then plot the boxplots.

In [ ]:
# YOUR CODE HERE


**Solution 19**

In [ ]:
employment_types = applicants['Employment_Type'].unique()
grouped_dti = [applicants.loc[applicants['Employment_Type'] == et, 'Debt_to_Income']
               for et in employment_types]

fig, ax = plt.subplots()
ax.boxplot(grouped_dti)
plt.show()

print(list(employment_types))


**Q20.** Repeat Q18 (the three risk-tier boxplots), and this time label the x-axis ticks using `ax.set_xticklabels(['Low', 'Medium', 'High'])` so the groups are identifiable without guessing the plotting order.

In [ ]:
# YOUR CODE HERE


**Solution 20**

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([low_dti, medium_dti, high_dti])
ax.set_xticklabels(['Low', 'Medium', 'High'])
ax.set_xlabel("Risk Tier")
ax.set_ylabel("Debt-to-Income Ratio (%)")
ax.set_title("Debt-to-Income by Risk Tier")
plt.show()


**Q21.** Repeat Q20, but pass `patch_artist=True` to `ax.boxplot()` so the boxes are filled, and set the fill color of each box to `'lightblue'` by looping over the returned dictionary's `['boxes']` and calling `.set_facecolor('lightblue')` on each.

In [ ]:
# YOUR CODE HERE


**Solution 21**

In [ ]:
fig, ax = plt.subplots()
bp = ax.boxplot([low_dti, medium_dti, high_dti], patch_artist=True)

for box in bp['boxes']:
    box.set_facecolor('lightblue')

ax.set_xticklabels(['Low', 'Medium', 'High'])
ax.set_xlabel("Risk Tier")
ax.set_ylabel("Debt-to-Income Ratio (%)")
plt.show()


**Q22.** Programmatically confirm what the boxplot shows visually: using the IQR method on `applicants['Debt_to_Income']` (`Q1`, `Q3`, `IQR = Q3 - Q1`, outlier bound `Q3 + 1.5 * IQR`), filter and print the rows of `applicants` whose `Debt_to_Income` exceeds that upper bound.

In [ ]:
# YOUR CODE HERE


**Solution 22**

In [ ]:
Q1 = applicants['Debt_to_Income'].quantile(0.25)
Q3 = applicants['Debt_to_Income'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

outliers = applicants[applicants['Debt_to_Income'] > upper_bound]
print(f"Upper bound: {upper_bound:.1f}")
print(outliers[['Debt_to_Income', 'Risk_Tier', 'Employment_Type']])


**Q23.** Create a Figure with 2 rows, 1 column. On the top Axes, plot a histogram of `applicants['Debt_to_Income']`. On the bottom Axes, plot a horizontal boxplot (`vert=False`) of the same data. Stacking these is a common way to see both the full shape *and* the exact outliers of one variable at once.

In [ ]:
# YOUR CODE HERE


**Solution 23**

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})

axes[0].hist(applicants['Debt_to_Income'], bins=30, edgecolor='black')
axes[0].set_title("Debt-to-Income: Distribution + Outliers")

axes[1].boxplot(applicants['Debt_to_Income'], vert=False)
axes[1].set_xlabel("Debt-to-Income Ratio (%)")

plt.show()


**Q24 (Capstone).** Build a two-panel risk data-quality dashboard. Left Axes: a labeled, styled histogram of `Credit_Score` (title, axis labels, gridlines). Right Axes: labeled boxplots of `Debt_to_Income` grouped by `Risk_Tier`, filled with `patch_artist=True`, with x-tick labels and a title. Add one `fig.suptitle()` covering both panels.

In [ ]:
# YOUR CODE HERE


**Solution 24**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: credit score histogram
axes[0].hist(applicants['Credit_Score'], bins=30, edgecolor='black', color='steelblue')
axes[0].set_title("Credit Score Distribution")
axes[0].set_xlabel("Credit Score")
axes[0].set_ylabel("Number of Applicants")
axes[0].grid(True, alpha=0.3)

# Right: debt-to-income by risk tier
bp = axes[1].boxplot([low_dti, medium_dti, high_dti], patch_artist=True,
                      flierprops=dict(marker='D', markerfacecolor='red', markersize=5))
for box in bp['boxes']:
    box.set_facecolor('lightblue')
axes[1].set_xticklabels(['Low', 'Medium', 'High'])
axes[1].set_xlabel("Risk Tier")
axes[1].set_ylabel("Debt-to-Income Ratio (%)")
axes[1].set_title("DTI Outliers by Risk Tier")

fig.suptitle("Applicant Pool: Data Quality Dashboard", fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()


---
## Checkpoint: Phase 2 Complete

You've covered:
- **Histograms** — `ax.hist()`, bin control (`bins`, custom edges), `density`, overlaying
  groups with `alpha`, marking the mean with `axvline()`, and spotting skew (e.g. income)
  and bad data entry (e.g. an implausible age) just by looking at the shape
- **Boxplots** — `ax.boxplot()`, reading quartiles/whiskers/fliers, grouped boxplots by
  category, customizing outlier markers with `flierprops`, filling boxes with
  `patch_artist=True`, and connecting what you see visually back to the IQR method you
  already know from pandas

Together, these two chart types are your primary outlier-detection workflow: histograms
tell you *how the data is shaped*, boxplots tell you *exactly which points don't belong*.

**Next up (Phase 3):** likely categorical comparison charts — bar charts for comparing
metrics across groups (e.g. default rate by employment type), and possibly stacked bars
or pie charts for composition. Let me know when you're ready!
